In [1]:
%pwd

'd:\\Siam\\Chicken-Disease-Classification\\research'

In [2]:
import os

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Siam\\Chicken-Disease-Classification'

In [5]:
import tensorflow as tf

In [7]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    params_image_size: list
    params_batch_size: int

In [8]:
from chicken_disease_classification.constants import *
from chicken_disease_classification.utils.common import read_yaml, create_directories

In [9]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config["artifacts_root"]])


    def get_evaluation_config(self) -> EvaluationConfig:
        config = self.config

        evaluation_config = EvaluationConfig(
            path_of_model=Path(config["training"]["trained_model_path"]),
            training_data=Path(config["data_validation"]["unzip_data_dir"]),
            all_params=self.params,
            params_image_size=self.params["IMAGE_SIZE"],
            params_batch_size=self.params["BATCH_SIZE"]
        )
        return evaluation_config


In [10]:
from chicken_disease_classification.utils.common import save_json
from chicken_disease_classification.exception import CustomException
import sys

In [11]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):
        datagenerator_kwargs = dict(
            rescale=1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )


    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)

    
    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

In [12]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.save_score()

except Exception as e:
   raise CustomException(e, sys)

[2026-05-10 17:05:45,838: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-10 17:05:45,841: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-10 17:05:45,843: INFO: common: created directory at: artifacts]
[2026-05-10 17:05:47,119: WARNING: config: TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.]
[2026-05-10 17:05:47,124: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Found 116 images belonging to 2 classes.
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 274ms/step - accuracy: 0.9569 - loss: 0.6277
[2026-05-10 17:05:51,717: INFO: common: json file saved at: scores.json]
